# 4A · The Grammar of Charts
### Financial Analytics — Module 4

A chart is a sentence. Before choosing *which* chart, name what the sentence must say. Seven intents cover almost everything:

| Intent | The sentence | The chart |
|---|---|---|
| **Trend** | "It moved like this over time" | Line |
| **Composition** | "It's made of these parts" | Stacked bar / area |
| **Distribution** | "The values spread like this" | Histogram / box |
| **Relationship** | "These two move together (or don't)" | Scatter |
| **Ranking** | "These are the biggest" | Sorted horizontal bar |
| **Uncertainty** | "The honest answer is a range" | Band / error bars |
| **Flow** | "It got from A to B via these steps" | Waterfall (notebook 4B) |

One rule above all: **one message per exhibit.** If a chart needs three sentences to explain, it's three charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")          # seaborn = matplotlib with better defaults
COLORS = ["#2563EB","#7C3AED","#0D9488","#EA580C","#DB2777","#16A34A"]

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
px  = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])
fin = pd.read_csv(BASE + "company_financials.csv")
cli = pd.read_csv(BASE + "client_book.csv")
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
print("Loaded:", px.shape, fin.shape, cli.shape, uni.shape)

---
## 1. TREND — the line chart

The message: *"MoneyMart grew steadily, dipped in COVID, then re-accelerated."* 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fin["fiscal_year"], fin["revenue_cr"], marker="o", color=COLORS[0], lw=2)

# Annotate the story ON the chart - don't make readers hunt for it
covid = fin[fin["fiscal_year"] == "FY20-21"]
ax.annotate("COVID dip", xy=(covid.index[0], covid["revenue_cr"].iloc[0]),
            xytext=(covid.index[0]-1.8, covid["revenue_cr"].iloc[0]+2200),
            arrowprops=dict(arrowstyle="->", color="#DC2626"), color="#DC2626")

ax.set_title("MoneyMart India — Revenue, FY17 to FY26 (Rs crore)", loc="left", fontweight="bold")
ax.set_ylabel("Rs crore")
plt.tight_layout(); plt.show()

Notice three deliberate choices: the title states the *finding* location and units; the annotation points at the story; the y-axis is labelled with units. A chart should survive being screenshotted into a WhatsApp message with zero explanation.

### ✏️ Exercise 1
Plot `stores_count` over the fiscal years. Give it a message-carrying title (not "Stores Count" — what *happened* to stores?).

In [ ]:
# your code here


---
## 2. COMPOSITION — the stacked bar

The message: *"Where does each rupee of revenue go?"* 

In [ ]:
cost_cols = ["cogs_cr", "employee_cost_cr", "marketing_cr", "other_opex_cr", "pat_cr"]
labels = ["COGS", "Employees", "Marketing", "Other opex", "Profit (PAT)"]

shares = fin[cost_cols].div(fin["revenue_cr"], axis=0) * 100   # % of revenue

fig, ax = plt.subplots(figsize=(10, 4.5))
bottom = np.zeros(len(fin))
for col, lab, clr in zip(cost_cols, labels, COLORS):
    ax.bar(fin["fiscal_year"], shares[col], bottom=bottom, label=lab, color=clr, width=0.7)
    bottom += shares[col].values

ax.set_title("Every Rs 100 of revenue: where it goes", loc="left", fontweight="bold")
ax.set_ylabel("% of revenue"); ax.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

(The bars don't reach 100 — the gap is depreciation, interest and tax. Spotting *what a composition chart leaves out* is itself an analyst skill.)

---
## 3. DISTRIBUTION — histogram and box

The message: *"Client AUM is wildly skewed — the average is a lie."* 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: raw AUM - unreadable because of skew
axes[0].hist(cli["aum_inr"].dropna(), bins=60, color=COLORS[0], alpha=0.8)
axes[0].set_title("AUM (raw) — a few giants crush the picture", loc="left", fontsize=10)

# Right: log scale reveals the true shape
axes[1].hist(np.log10(cli["aum_inr"].dropna()), bins=40, color=COLORS[2], alpha=0.8)
axes[1].set_title("AUM (log10 scale) — now the shape is visible", loc="left", fontsize=10)
axes[1].set_xlabel("log10(AUM)  [5 = Rs 1 lakh, 7 = Rs 1 crore]")
plt.tight_layout(); plt.show()

print(f"Mean AUM   : Rs {cli['aum_inr'].mean():>12,.0f}   <- dragged up by a few Ultra-HNI")
print(f"Median AUM : Rs {cli['aum_inr'].median():>12,.0f}   <- the typical client")

**Mean vs median is a descriptive-analytics decision, not a math detail.** For skewed money data (AUM, income, ticket sizes) the median describes the typical case; the mean describes the total divided by heads. Report the one that answers the question asked — and say which you used.

### ✏️ Exercise 2
Draw a box plot of `aum_inr` **by segment** (`sns.boxplot(data=cli, x="segment", y="aum_inr")`). Add `ax.set_yscale("log")`. Which segment has the widest spread?

In [ ]:
# your code here


---
## 4. RELATIONSHIP — the scatter

The message: *"Do clients with more products hold more AUM?"* 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=cli, x="products_held", y="aum_inr", hue="segment",
              palette=COLORS[:4], alpha=0.5, jitter=0.25, ax=ax)
ax.set_yscale("log")
ax.set_title("Products held vs AUM — related, but segment does the heavy lifting",
             loc="left", fontweight="bold", fontsize=11)
plt.tight_layout(); plt.show()

A relationship chart's honest job includes showing when the relationship is **weak** or driven by a third variable (here, segment). Module 5 turns this instinct into a method — and Module 1's warning stands: relationship is not causation.

---
## 5. RANKING — the sorted horizontal bar

The message: *"Which sectors dominate our stock universe by traded value?"* 

In [ ]:
uni["traded_value"] = uni["close"] * uni["volume"]
sector_val = (uni.groupby("sector")["traded_value"].sum() / 1e12).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(sector_val.index, sector_val.values, color=COLORS[0])
ax.set_title("Traded value by sector (Rs lakh crore, 2022-25)", loc="left", fontweight="bold")
for i, v in enumerate(sector_val.values):
    ax.text(v, i, f" {v:,.1f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

Ranking rules: horizontal (labels stay readable), **sorted** (unsorted ranking bars are a chart crime), values labelled on the bars.

---
## 6. UNCERTAINTY — the band

The message: *"The recent average is a line; the truth is a range."* 

In [ ]:
px = px.sort_values("date")
px["ma20"] = px["close"].rolling(20).mean()
px["sd20"] = px["close"].rolling(20).std()
recent = px.tail(250)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(recent["date"], recent["close"], lw=0.8, color="#94A3B8", label="Close")
ax.plot(recent["date"], recent["ma20"], lw=1.6, color=COLORS[0], label="20-day average")
ax.fill_between(recent["date"], recent["ma20"]-2*recent["sd20"], recent["ma20"]+2*recent["sd20"],
                alpha=0.15, color=COLORS[0], label="±2 std dev band")
ax.set_title("NIFTY 50, last 250 sessions — showing the range, not just the line",
             loc="left", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Bands communicate honesty. In Module 6 this becomes doctrine: **a forecast without an interval is a guess wearing a suit.**

### ✏️ Exercise 3
The chart crime gallery: take your Exercise 1 stores chart and deliberately ruin it — start the y-axis at 150 instead of 0 to exaggerate growth. Look at both versions. Write one sentence on why truncated bar/area axes mislead (truncating a **line** chart's axis is often fine; truncating a **bar** chart's is lying — bars encode value by length).

---
## Recap
Seven intents, one message per exhibit, title states the finding, label the units, sort your rankings, show ranges when ranges are the truth. **Next: 4B — the exhibits that are specifically financial.**

*AI disclosure: ______*

In [ ]:
# workspace
